[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/rewrite/rewrite/examples/mechanics/stiffness.ipynb) [![Open in GitHub Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/EelcoHoogendoorn/numga?ref=rewrite)

# Planar Rigid-Body Stiffness & Vibration Modes in PGA2D

When a rigid body is suspended by elastic springs, our goal is to compute restoring forces and find natural vibration frequencies and mode shapes.

Rather than assembling coordinate stiffness matrices and moment arms by hand, this notebook uses **extensors** (linear maps between blade subspaces):
spring lines directly measure displacement into extension, Hooke's law compiles into a rank-1 stiffness extensor, and normal vibration modes emerge from a single generalized eigensolve between stiffness and inertia.


In [ ]:
# Setup environment: clone repository and configure paths if running in Colab
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    import os, shutil, subprocess
    os.chdir("/content")
    repo_dir = Path("/content/numga_repo")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", "-b", "rewrite", "https://github.com/EelcoHoogendoorn/numga.git", str(repo_dir)], check=True)
    else:
        subprocess.run(["git", "-C", str(repo_dir), "pull", "origin", "rewrite"], check=False)
    rewrite_dir = repo_dir / "rewrite"
    src_dir = rewrite_dir / "src"
    for p in [str(src_dir), str(rewrite_dir)]:
        if p not in sys.path:
            sys.path.insert(0, p)

# Ensure rewrite root is in sys.path when running locally:
for cand in [Path.cwd(), *Path.cwd().parents]:
    if (cand / "examples" / "mechanics").is_dir():
        if str(cand) not in sys.path:
            sys.path.insert(0, str(cand))
        break
    if (cand / "rewrite" / "examples" / "mechanics").is_dir():
        p = str(cand / "rewrite")
        if p not in sys.path:
            sys.path.insert(0, p)
        break


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Image
from pathlib import Path

from numga import NumpyContext
from numga.algebras import PGA2D
from examples.mechanics.stiffness_plumbing import (
    render_setup,
    render_modes,
    render_animation,
)

# Bind 2D Projective Geometric Algebra (PGA2D):
ctx = NumpyContext(PGA2D)
mv = ctx.multivector

# Blade Subspaces:
Scalar = PGA2D.gatype.scalar()              # Grade 0: scalar real values
Point = PGA2D.gatype.antivector()           # Grade 2: projective points in the 2D plane
Twist = PGA2D.gatype.bivector()             # Grade 2 dual / Lie algebra: infinitesimal rigid motions & velocities
Wrench = PGA2D.gatype.vector()              # Grade 1: lines of action for forces, torques, and springs

# Extensors (linear maps between blade subspaces):
SpringExtension = PGA2D.gatype((Scalar, Twist))   # Scalar <- Twist: measures spring extension from body motion
Stiffness = PGA2D.gatype((Wrench, Twist))         # Wrench <- Twist: restoring force and torque from body motion
Inertia = PGA2D.gatype((Wrench, Twist))           # Wrench <- Twist: kinetic momentum wrench from twist velocity

def point(xy: np.ndarray) -> Point:
    """Embed (..., 2) Cartesian positions as unit-weight PGA points."""
    xy = np.asarray(xy, dtype=float)
    return mv.antivector(np.concatenate([xy, np.ones_like(xy[..., :1])], axis=-1))

print("PGA2D Algebra and Extensor types initialized successfully.")


## 1. Rigid Body Geometry & Spring Lines

A uniform 2x1 rectangular plate of mass 1 is suspended in the plane by elastic springs attached to fixed anchors.

In PGA, a spring is simply the line joining its anchor to its attachment point: `lines = (anchors & attachments).normalized()`. Below, we set up both suspensions (two vertical springs vs. an added angled spring) and plot the physical system at equilibrium.


In [ ]:
# 1. Uniform 2x1 rigid plate vertices:
body = point([[-1.0, -0.5], [1.0, -0.5], [1.0, 0.5], [-1.0, 0.5]])

# 2. Case A: Two vertical springs (anchors directly above attachments):
att_par = point([[-0.8, 0.5], [0.8, 0.5]])
anc_par = point([[-0.8, 1.55], [0.8, 1.55]])
k_par = mv.scalar(np.full((2, 1), 6.0))

# 3. Case B: Two vertical springs + one off-centre angled spring:
att_ang = point([[-0.8, 0.5], [0.8, 0.5], [1.0, 0.0]])
anc_ang = point([[-0.8, 1.55], [0.8, 1.55], [1.9, 0.85]])
k_ang = mv.scalar(np.full((3, 1), 6.0))

# 4. Spring lines of action in PGA (joining anchor to attachment):
lines_par = (anc_par & att_par).normalized()   # [2] Wrench
lines_ang = (anc_ang & att_ang).normalized()   # [3] Wrench

# Plot 1: Setup - Rigid plate suspended by springs at equilibrium
fig = render_setup(
    body=body,
    anchors_list=[anc_par, anc_ang],
    attachments_list=[att_par, att_ang],
    titles=["Case A: Two Vertical Springs", "Case B: Added Angled Spring"],
    plot_path="examples/plots/stiffness_setup.png",
)
plt.show()


## 2. Building the Stiffness Extensor from Springs

Hooke's law states that restoring force is proportional to extension along the spring's line of action.

In PGA, pairing a spring line with an open `Twist` measures extension: `extension = Twist & lines`. Multiplying by the line and spring constant compiles directly into the stiffness extensor: `stiffness = (lines * extension * k).sum()`.


In [ ]:
# 1. Measure spring extension from an open rigid-body motion (linear form: Twist -> Scalar):
extension_par = Twist & lines_par
extension_ang = Twist & lines_ang

# 2. Compile Hooke's law into stiffness extensors mapping body displacement to opposing wrench:
stiffness_par = (lines_par * extension_par * k_par).sum(axis=0)
stiffness_ang = (lines_ang * extension_ang * k_ang).sum(axis=0)

print("Stiffness extensor GAType:", stiffness_par.gatype)
print("Kernel shape             :", stiffness_par.kernel.shape, " # 3x3 extensor: Wrench <- Twist")
print("Stiffness matrix (Case A - Two vertical springs):\n", np.round(stiffness_par.kernel, 3))
print("\nStiffness matrix (Case B - Added angled spring):\n", np.round(stiffness_ang.kernel, 3))


## 3. Building the Inertia Extensor from Mass Points

The inertia extensor maps twist velocity to kinetic momentum wrench (`Wrench <- Twist`).

Each mass point contributes a rate-to-momentum extensor: `p & (p.commutator(Twist)) * m`. 4 Gauss quadrature points capture the plate's mass (1.0 kg) and polar moment of inertia (5/12 kg*m^2) exactly.


In [ ]:
# 1. Mass distribution via 2-point Gauss quadrature on the 2x1 rectangle:
mass_coords = np.array([[-1.0, -0.5], [1.0, -0.5], [1.0, 0.5], [-1.0, 0.5]]) / np.sqrt(3)
mass_points = point(mass_coords)
masses = mv.scalar(np.full((4, 1), 0.25))

# 2. Compile inertia extensor mapping twist velocity to momentum wrench:
inertia = (mass_points & mass_points.commutator(Twist) * masses).sum(axis=0)

print("Inertia extensor GAType:", inertia.gatype)
print("Kernel shape           :", inertia.kernel.shape, " # 3x3 extensor: Wrench <- Twist")
print("Inertia matrix:\n", np.round(inertia.kernel, 3))


## 4. Modal Analysis via Generalized Bilinear Eigensolve

Natural vibration modes solve the generalized eigenvalue problem between elastic potential energy and kinetic energy.

In `numga`, `(Twist & stiffness).eigh(Twist & inertia)` solves this directly on the two bilinear forms without matrix inversion. The eigenvalues give natural frequencies, while eigenvectors give normal mode twists.


In [ ]:
# 1. Solve generalized eigenvalue problem directly on bilinear energy forms:
#    (Twist & stiffness) is the potential energy form; (Twist & inertia) is the kinetic energy form:
values_par, modes_par = (Twist & stiffness_par).eigh(Twist & inertia)
values_ang, modes_ang = (Twist & stiffness_ang).eigh(Twist & inertia)

# 2. Natural frequencies in Hz: f = sqrt(lambda) / (2 * pi)
freqs_par = np.sqrt(np.maximum(values_par.kernel[..., 0], 0.0)) / (2 * np.pi)
freqs_ang = np.sqrt(np.maximum(values_ang.kernel[..., 0], 0.0)) / (2 * np.pi)

print("=== Case A: Two Vertical Springs ===")
for label, freq in zip(["Free slide", "Bounce", "Rock"], freqs_par):
    detail = "0.00 Hz (free mechanism)" if freq == 0 else f"{freq:.3f} Hz"
    print(f"  {label:<16}: {detail}")

print("\n=== Case B: Two Vertical + One Angled Spring ===")
for label, freq in zip(["Coupled mode 1", "Coupled mode 2", "Coupled mode 3"], freqs_ang):
    print(f"  {label:<16}: {freq:.3f} Hz")


## 5. Visualizing Natural Vibration Modes

Each row shows the three independent small-motion modes of the rigid body.

Displacements are amplified for clarity. Coiled springs are color-coded: orange for lengthening, blue for shortening, and grey for unchanged. In the top row, sideways motion leaves both vertical springs unchanged to first order (a 0 Hz free slide).


In [ ]:
# Plot 2: 6-Panel Mode Comparison - Small-motion modes and spring extensions
fig = render_modes(
    body=body,
    anchors_list=[anc_par, anc_ang],
    attachments_list=[att_par, att_ang],
    modes_list=[modes_par, modes_ang],
    values_list=[values_par, values_ang],
    extensions_list=[extension_par, extension_ang],
    plot_path="examples/plots/stiffness.png",
)
plt.show()


## 6. Harmonic Oscillation Animation

Releasing the plate from rest in each mode demonstrates physical time evolution.

Zero-frequency modes remain statically displaced, while non-zero modes oscillate at their computed natural frequencies. All panels share synchronized physical time.


In [ ]:
# Plot 3: Synchronized Harmonic Vibration Animation
gif_path = Path("examples/plots/stiffness.gif")
render_animation(
    body=body,
    anchors_list=[anc_par, anc_ang],
    attachments_list=[att_par, att_ang],
    modes_list=[modes_par, modes_ang],
    values_list=[values_par, values_ang],
    extensions_list=[extension_par, extension_ang],
    animation_path=str(gif_path),
)

if gif_path.exists():
    display(Image(filename=str(gif_path)))


## 7. Summary & Conceptual Synthesis

Having walked through the working pipeline, we can step back and examine how **extensors** structure rigid-body mechanics:

* **Measuring Deformation with Lines (Spring Extension as a Linear Form)**:
  A spring is a line joining an anchor to an attachment point (`lines = anchors & attachments`). Pairing that line with an open twist (`Twist & lines`) directly yields a linear form (`Twist -> Scalar`) measuring spring stretch under any rigid body motion without coordinate projection formulas.

* **Hooke's Law as Rank-1 Extensor Dyads (Stiffness Extensor)**:
  Multiplying the line of action by its extension form and spring constant (`lines * (Twist & lines) * k`) forms a rank-1 extensor dyad mapping body displacements to restoring force and torque (`Wrench <- Twist`). Summing across springs yields the complete stiffness extensor.

* **Kinetic Mass Distribution (Inertia Extensor)**:
  Each mass point contributes a rate-to-momentum extensor: `p & (p.commutator(Twist)) * m`. Summing across mass points integrates total mass, center of mass, and rotational inertia into a single geometric extensor (`Wrench <- Twist`).

* **Solving Modes Directly on Bilinear Energy Forms (Generalized Eigensolve)**:
  Rather than building coordinate mass and stiffness matrices, the vibration eigenvalue problem is solved directly between the potential energy form and kinetic energy form: `(Twist & stiffness).eigh(Twist & inertia)`.

* **Evaluating Physical Displacements (Lie Algebra Commutators)**:
  The motion of any point on the rigid body under an eigenvector twist step is evaluated directly via the Lie algebra commutator (`body.commutator(modes)`), cleanly connecting abstract Lie algebra coordinates to physical geometry.
